# Phase 1 — Airline Delay Exploratory Analysis
Reconstructed from the Phase 1 presentation. Supply a Data Expo 2009 airline CSV (e.g. `2003.csv`) in `data/`. The notebook does not include original data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
DATA = Path("../data/2003.csv")
if not DATA.exists():
    raise FileNotFoundError(f"Download the 2003 airline dataset and place it at {DATA}")
df = pd.read_csv(DATA, na_values=["\\N", "NA"], low_memory=False)
print(df.shape); display(df.head())

In [ ]:
required = ["Origin", "Dest", "UniqueCarrier", "ArrDelay", "DepDelay"]
missing = [c for c in required if c not in df]
if missing: raise ValueError(f"Missing columns: {missing}")
for c in ["ArrDelay", "DepDelay"]: df[c] = pd.to_numeric(df[c], errors="coerce")
# Sum positive delay minutes only; negative values mean early arrival/departure.
for c in ["ArrDelay", "DepDelay"]: df[c + "Positive"] = df[c].clip(lower=0)

In [ ]:
airport = (df.groupby("Origin")[["ArrDelayPositive", "DepDelayPositive"]].sum().sum(axis=1) / 60).nlargest(3)
carrier = (df.groupby("UniqueCarrier")[["ArrDelayPositive", "DepDelayPositive"]].sum().sum(axis=1) / 60).nlargest(3)
comparison = df[["ArrDelayPositive", "DepDelayPositive"]].sum() / 60
print("Top origin airports (total positive delay hours):"); display(airport)
print("Top carriers (total positive delay hours):"); display(carrier)
print("Arrival vs departure delay hours:"); display(comparison)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,4))
airport.plot.bar(ax=ax, title="Top 3 origin airports — delay hours")
ax.set_ylabel("Hours"); plt.tight_layout(); plt.show()

**Interpretation note:** The presentations do not consistently specify whether early arrivals/departures were included. Here negative delays are clipped at zero, and total positive arrival + departure delay is used for rankings. Adjust this definition if your original assignment used a different metric.